In [11]:
# Standard libraries
import os
import sys
from pathlib import Path
import glob
import json
import re

# =========================
# Numerical computing
# =========================
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.constants as const
from scipy.interpolate import interp1d, RegularGridInterpolator
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter

# =========================
# Plotting
# =========================
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

# =========================
# HDF5 / MATLAB / Lumerical data
# =========================
import h5py

# =========================
# Notebook display helpers
# =========================
from IPython.display import display, Markdown

In [12]:
# Plot style
# =========================
plt.rcParams.update({
    "figure.figsize": (6, 4),
    "font.family": "Arial",
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 12,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "lines.linewidth": 2,
})

# Useful constants
c0 = const.c
eps0 = const.epsilon_0
mu0 = const.mu_0

# Plot the broadband focusing of the system #


In [ ]:
data = np.load("/Users/yiyangzhi/Library/CloudStorage/GoogleDrive-yiyang_zhi3@berkeley.edu/My Drive/Ming Wu Integrated Photonics Group/Paper/Gen1_Photonics/Figures/FIG2/broadband_focusing_extracted.npz")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, rgb_to_hsv


# ============================================================
# Load extracted panel(a) data
# ============================================================

wavelengths = data["wavelengths_nm"].astype(int)
rgb_uint8 = data["rgb_uint8"]      # shape: (8, H, W, 3), exact displayed RGB

# Recompute scalar intensity directly from RGB to avoid NaN issues
rgb_float = rgb_uint8.astype(float) / 255.0
I_raw = np.max(rgb_float, axis=-1)

# Normalize intensity panel-by-panel
I_norm = np.zeros_like(I_raw)

for k in range(I_raw.shape[0]):
    Ik = I_raw[k]
    Ik = Ik - np.nanmin(Ik)

    max_val = np.nanmax(Ik)
    if max_val > 0:
        Ik = Ik / max_val

    I_norm[k] = Ik


# ============================================================
# Robust colormap / peak-color recovery
# ============================================================

def recover_peak_color(rgb_panel_uint8):
    """
    Recover the dominant saturated beam color from one RGB panel.

    This rejects:
    - black background
    - gray / white pixels
    - very dim pixels
    - weakly saturated pixels
    """

    rgb = rgb_panel_uint8.astype(float) / 255.0
    flat_rgb = rgb.reshape(-1, 3)

    hsv = rgb_to_hsv(flat_rgb.reshape(1, -1, 3)).reshape(-1, 3)

    hue = hsv[:, 0]
    sat = hsv[:, 1]
    val = hsv[:, 2]

    # Main mask: bright and strongly colored pixels
    valid = (
        np.isfinite(val)
        & (val > np.percentile(val, 75))
        & (sat > 0.35)
    )

    # Relax if too few pixels are found
    if np.sum(valid) < 20:
        valid = (
            np.isfinite(val)
            & (val > np.percentile(val, 60))
            & (sat > 0.20)
        )

    # Final fallback
    if np.sum(valid) < 5:
        valid = (
            np.isfinite(val)
            & (val > np.percentile(val, 50))
            & (sat > 0.10)
        )

    if np.sum(valid) == 0:
        raise RuntimeError("No valid colored pixels found.")

    # Score favors saturated and bright pixels
    score = sat * val

    valid_score = score[valid]

    # Use top colored pixels, not necessarily the absolute brightest pixels
    cutoff = np.percentile(valid_score, 98)
    selected = valid & (score >= cutoff)

    if np.sum(selected) < 5:
        cutoff = np.percentile(valid_score, 95)
        selected = valid & (score >= cutoff)

    selected_rgb = flat_rgb[selected]

    # Normalize each selected pixel by its own maximum channel
    selected_rgb = selected_rgb / np.maximum(
        selected_rgb.max(axis=1, keepdims=True),
        1e-12,
    )

    # Robust representative color
    peak_rgb = np.median(selected_rgb, axis=0)

    # Normalize so the strongest channel is 1
    peak_rgb = peak_rgb / np.maximum(np.max(peak_rgb), 1e-12)

    return peak_rgb

peak_rgb = {}

for k, wl in enumerate(wavelengths):
    try:
        peak_rgb[wl] = recover_peak_color(rgb_uint8[k])
    except RuntimeError:
        print(f"Warning: could not recover color for {wl} nm. Using fallback.")
        fallback_colors = {
            405: np.array([0.23, 0.00, 1.00]),
            450: np.array([0.02, 0.10, 1.00]),
            515: np.array([0.00, 1.00, 0.35]),
            635: np.array([1.00, 0.00, 0.05]),
            730: np.array([1.00, 0.13, 0.06]),
            830: np.array([0.92, 0.43, 1.00]),
            880: np.array([0.78, 0.50, 1.00]),
            940: np.array([0.75, 0.45, 1.00]),
        }
        peak_rgb[wl] = fallback_colors[wl]

selected_wavelengths = np.array([405, 450, 515, 635, 730, 880])
selected_indices = [np.where(wavelengths == wl)[0][0] for wl in selected_wavelengths]

print("Recovered peak colors:")
for wl in selected_wavelengths:
    color = peak_rgb[wl]
    color_uint8 = np.clip(np.round(color * 255), 0, 255).astype(int)
    hex_color = "#{:02x}{:02x}{:02x}".format(*color_uint8)
    print(f"{wl} nm: {hex_color}, RGB = {color}")

fig, axes = plt.subplots(
    2, 3,
    figsize=(8.8, 5.8),
    dpi=220,
    constrained_layout=True,
)

for ax, idx, wl in zip(axes.ravel(), selected_indices, selected_wavelengths):

    ax.imshow(
        rgb_uint8[idx],
        extent=[-30, 30, -30, 30],
        origin="upper",
        interpolation="nearest",
    )

    ax.set_title(rf"$\lambda$ = {wl} nm", fontsize=18)
    ax.set_xlim(-30, 30)
    ax.set_ylim(-30, 30)
    ax.set_aspect("equal")

    ax.set_xticks([-30, -20, -10, 0, 10, 20, 30])
    ax.set_yticks([-30, -20, -10, 0, 10, 20, 30])

    ax.tick_params(axis="both", labelsize=18)

plt.show()





